In [ ]:
# Cell 1 — Mount Drive, install packages, create folders, verify source files
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/PhishGuard'

import os, sys
for folder in ['data', 'data/checkpoints', 'data/sources',
               'features', 'models', 'evaluation']:
    os.makedirs(f'{DRIVE_BASE}/{folder}', exist_ok=True)

%pip install requests pandas openpyxl web3 tqdm -q

required = [
    'eth_labels_phishing.csv',
    'eth_labels_exchange.csv',
    'eth_labels_token_contracts.csv',
    'fraud_contracts.csv',
    'kaggle_fraud.csv',
    'scamsniffer_addresses.json',
    'mew_darklist.json',
    'PTXPHISH.xlsx',
    'defi_seeds.json'
]
missing = [f for f in required
           if not os.path.exists(f'{DRIVE_BASE}/data/sources/{f}')]
if missing:
    print(f'MISSING FILES: {missing}')
    print(f'Upload all missing files to: {DRIVE_BASE}/data/sources/')
    sys.exit()
print('All 9 source files confirmed.')
print('Cell 1 ready.')


In [ ]:
# Cell 2: Constants
import random

ETHERSCAN_API_KEY     = 'WI9RMPKI74VKDCZ8PHUYF1KK36RV8CVZN2'
ETHERSCAN_BASE        = 'https://api.etherscan.io/v2/api'
ETHERSCAN_CHAIN_ID    = 1
APPROVAL_TOPIC        = ('0x8c5be1e5ebec7d5bd14f71427d1e84f3dd0314c0f7b2291'
                         'e5b200ac8c7c3b925')
CLOUDFLARE_RPC        = 'https://cloudflare-eth.com'
ALCHEMY_RPC           = 'https://rpc.ankr.com/eth'
SLEEP                 = 0.3
VALID_ADDR_RE         = r'^0x[0-9a-fA-F]{40}$'
WALLET_ACTIVITY_MIN   = 5
CONTRACT_ACTIVITY_MIN = 2
TARGET_PER_CLASS      = 2000
CONTRACT_TARGET       = 650

DRIVE_BASE = '/content/drive/MyDrive/PhishGuard'

random.seed(42)
print('Constants loaded.')


In [ ]:
import time, random, json, os, requests
from tqdm.auto import tqdm

In [ ]:
# Cell 3: Load all 9 source files with data quality fixes
import pandas as pd, json, re, requests

def is_valid(addr):
    return bool(re.match(VALID_ADDR_RE, str(addr).strip()))

# ── File 1: eth_labels_phishing ─────────────────────────────
df_phish = pd.read_csv(f'{DRIVE_BASE}/data/sources/eth_labels_phishing.csv')
df_phish = df_phish[df_phish['address'].str.strip().apply(is_valid)].copy()
df_phish['address'] = df_phish['address'].str.strip().str.lower()
print(f'[1] eth_labels_phishing:        {len(df_phish):,} valid')

# ── File 2: eth_labels_exchange ──────────────────────────────
df_exch = pd.read_csv(f'{DRIVE_BASE}/data/sources/eth_labels_exchange.csv')
df_exch['address'] = df_exch['address'].str.strip().str.lower()
print(f'[2] eth_labels_exchange:        {len(df_exch):,}')

# ── File 3: eth_labels_token_contracts ───────────────────────
df_tok = pd.read_csv(
    f'{DRIVE_BASE}/data/sources/eth_labels_token_contracts.csv',
    on_bad_lines='skip')
df_tok['address'] = df_tok['address'].str.strip().str.lower()
df_tok = df_tok[df_tok['address'].apply(is_valid)].drop_duplicates('address').copy()
print(f'[3] eth_labels_token_contracts: {len(df_tok):,}')

# ── File 4: fraud_contracts ──────────────────────────────────
df_fraud = pd.read_csv(f'{DRIVE_BASE}/data/sources/fraud_contracts.csv')
df_fraud['address'] = df_fraud['address'].str.strip().str.lower()
print(f'[4] fraud_contracts:            {len(df_fraud):,}')

# ── File 5: kaggle_fraud ─────────────────────────────────────
df_kag = pd.read_csv(f'{DRIVE_BASE}/data/sources/kaggle_fraud.csv')
df_kag = df_kag[df_kag['Address'].apply(is_valid)].drop_duplicates('Address').copy()
df_kag['Address'] = df_kag['Address'].str.lower()
kag_phish  = set(df_kag[df_kag['FLAG']==1]['Address'])
kag_benign = set(df_kag[df_kag['FLAG']==0]['Address'])
print(f'[5] kaggle phishing:            {len(kag_phish):,}')
print(f'    kaggle benign:              {len(kag_benign):,}')

# ── File 6: scamsniffer_addresses.json ───────────────────────
with open(f'{DRIVE_BASE}/data/sources/scamsniffer_addresses.json') as f:
    scam_raw = json.load(f)
scam_set = set(a.lower() for a in scam_raw if is_valid(a))
print(f'[6] scamsniffer:                {len(scam_set):,}')

# ── File 7: mew_darklist.json ────────────────────────────────
with open(f'{DRIVE_BASE}/data/sources/mew_darklist.json') as f:
    mew_raw = json.load(f)
mew_set = set(e['address'].lower() for e in mew_raw if is_valid(e.get('address', '')))
print(f'[7] mew_darklist:               {len(mew_set):,}')

# ── File 8: PTXPHISH.xlsx ────────────────────────────────────
xl        = pd.read_excel(f'{DRIVE_BASE}/data/sources/PTXPHISH.xlsx', header=None)
data_rows = xl.iloc[4:]

ptx_phish_src = set()
for col in [22, 24]:
    for v in data_rows[col].dropna().astype(str):
        for m in re.findall(r'etherscan\.io/address/(0x[0-9a-fA-F]{40})', v, re.I):
            ptx_phish_src.add(m.lower())

ptx_legit_src = set()
for col in [1, 3, 5, 8, 10, 12]:
    for v in data_rows[col].dropna().astype(str):
        for m in re.findall(r'etherscan\.io/address/(0x[0-9a-fA-F]{40})', v, re.I):
            ptx_legit_src.add(m.lower())

overlap_ptx     = ptx_phish_src & ptx_legit_src
ptx_phish_clean = ptx_phish_src - overlap_ptx

ptx_tx_hashes = set()
for col in [14, 16, 18, 21, 23]:
    for v in data_rows[col].dropna().astype(str):
        v = v.strip()
        if re.match(r'^0x[0-9a-fA-F]{64}$', v):
            ptx_tx_hashes.add(v.lower())

print(f'[8] PTXPHISH phishing contracts:{len(ptx_phish_clean):,}')
print(f'    PTXPHISH tx hashes:         {len(ptx_tx_hashes):,}')
print(f'    PTXPHISH legit contracts:   {len(ptx_legit_src):,}')

# ── File 9: defi_seeds.json ──────────────────────────────────
with open(f'{DRIVE_BASE}/data/sources/defi_seeds.json') as f:
    defi_seeds_raw = json.load(f)
defi_seeds = [a.lower() for a in defi_seeds_raw if is_valid(a)]
print(f'[9] defi_seeds:                 {len(defi_seeds):,}')

print()
print('All 9 source files loaded successfully.')

# ── Extra phishing contract sources (live fetch) ─────────────
extra_pc_raw = set()

try:
    r = requests.get(
        'https://raw.githubusercontent.com/scamsniffer/scam-database/main/blacklist/contract.json',
        timeout=15)
    if r.status_code == 200:
        before = len(extra_pc_raw)
        for addr in r.json():
            if is_valid(addr):
                extra_pc_raw.add(addr.lower())
        print(f'Source A — ScamSniffer contracts:  +{len(extra_pc_raw)-before:,} addresses')
    else:
        print(f'Source A — ScamSniffer contracts:  HTTP {r.status_code}, skipping')
except Exception as e:
    print(f'Source A — ScamSniffer contracts:  failed ({e}), skipping')

try:
    r = requests.get(
        'https://raw.githubusercontent.com/scamsniffer/scam-database/main/blacklist/address.json',
        timeout=15)
    if r.status_code == 200:
        before = len(extra_pc_raw)
        for addr in r.json():
            if is_valid(addr):
                extra_pc_raw.add(addr.lower())
        print(f'Source B — ScamSniffer addresses:  +{len(extra_pc_raw)-before:,} addresses')
    else:
        print(f'Source B — ScamSniffer addresses:  HTTP {r.status_code}, skipping')
except Exception as e:
    print(f'Source B — ScamSniffer addresses:  failed ({e}), skipping')

try:
    r = requests.get(
        'https://raw.githubusercontent.com/CryptoScamDB/blacklist/master/data/urls.yaml',
        timeout=15)
    if r.status_code == 200:
        before = len(extra_pc_raw)
        for addr in re.findall(r'"(0x[0-9a-fA-F]{40})"', r.text):
            extra_pc_raw.add(addr.lower())
        print(f'Source C — CryptoScamDB YAML:      +{len(extra_pc_raw)-before:,} addresses')
    else:
        print(f'Source C — CryptoScamDB YAML:      HTTP {r.status_code}, skipping')
except Exception as e:
    print(f'Source C — CryptoScamDB YAML:      failed ({e}), skipping')

print(f'\nTotal extra phishing contract candidates: {len(extra_pc_raw):,}')


In [ ]:
# Cell 4: Build address pools
# ── Phishing wallet pool ─────────────────────────────────────
pw_raw = set()
pw_raw.update(df_phish['address'])
pw_raw.update(kag_phish)
pw_raw.update(scam_set)
pw_raw.update(mew_set)

# ── Phishing contract pool ───────────────────────────────────
pc_raw = set()
pc_raw.update(df_fraud['address'])
pc_raw.update(ptx_phish_clean)
pc_raw.update(extra_pc_raw)

# Remove confirmed contracts from wallet pool
overlap_wp_pc = pw_raw & pc_raw
pw_raw -= overlap_wp_pc
print(f'Removed {len(overlap_wp_pc)} confirmed contracts from wallet pool')

# ── Benign wallet pool ───────────────────────────────────────
bw_raw = set()
bw_raw.update(df_exch['address'])
bw_raw.update(kag_benign)

# ── Benign contract pool ─────────────────────────────────────
bc_raw = set()
bc_raw.update(df_tok['address'])
bc_raw.update(ptx_legit_src)

# ── DeFi seeds ───────────────────────────────────────────────
try:
    before_seed = len(bc_raw)
    for addr in defi_seeds:
        if addr not in pw_raw and addr not in pc_raw:
            bc_raw.add(addr)
    print(f'[DeFi seeds] Added {len(bc_raw) - before_seed} new addresses to bc_raw')
    print(f'[DeFi seeds] bc_raw now: {len(bc_raw):,}')
except Exception as e:
    print(f'[DeFi seeds] WARNING: seed block failed ({e}), continuing without seeds')

# ── Cross-contamination clean ────────────────────────────────
bw_raw -= pw_raw
bw_raw -= pc_raw
bc_raw -= pc_raw
bc_raw -= pw_raw

print(f'Raw pool sizes:')
print(f'  Phishing wallets:   {len(pw_raw):,}')
print(f'  Benign wallets:     {len(bw_raw):,}')
print(f'  Phishing contracts: {len(pc_raw):,}')
print(f'  Benign contracts:   {len(bc_raw):,}')

assert len(pw_raw & bw_raw) == 0, 'pw & bw contamination'
assert len(pw_raw & bc_raw) == 0, 'pw & bc contamination'
assert len(pc_raw & bw_raw) == 0, 'pc & bw contamination'
assert len(pc_raw & bc_raw) == 0, 'pc & bc contamination'
assert len(pw_raw & pc_raw) == 0, 'pw & pc contamination'
print('Cross-contamination check: PASSED')


In [ ]:
# Cell 5: Resolve PTXPHISH transaction hashes
# Each tx hash is resolved to its TO address via eth_getTransactionByHash.
# PTXPHISH hashes are calls TO existing phishing contracts (not deployments),
# so the phishing contract address is in the transaction's 'to' field.
# Checkpoint saves both processed hashes and resolved addresses so
# the session can resume correctly without reprocessing completed hashes.

ptx_ckpt_path = f'{DRIVE_BASE}/data/checkpoints/ptx_resolved.json'

if os.path.exists(ptx_ckpt_path):
    with open(ptx_ckpt_path) as f:
        ckpt = json.load(f)
    processed_hashes = set(ckpt.get('processed_hashes', []))
    ptx_resolved     = set(ckpt.get('resolved_addrs',   []))
    print(f'Resumed: {len(processed_hashes):,} hashes already processed, '
          f'{len(ptx_resolved):,} addresses resolved')
else:
    processed_hashes = set()
    ptx_resolved     = set()
    print('Starting fresh tx hash resolution')

def resolve_tx_hash(tx_hash):
    '''
    Calls Etherscan eth_getTransactionByHash via V2 API.
    Returns the TO address (the phishing contract that was called) or None.
    Never raises.
    '''
    try:
        r = requests.get(ETHERSCAN_BASE, params={
            'chainid': ETHERSCAN_CHAIN_ID,
            'module':  'proxy',
            'action':  'eth_getTransactionByHash',
            'txhash':  tx_hash,
            'apikey':  ETHERSCAN_API_KEY
        }, timeout=10)
        result = r.json().get('result', {})
        if not isinstance(result, dict):
            return None
        to_addr = result.get('to', '') or ''
        return to_addr.lower() if is_valid(to_addr) else None
    except Exception:
        return None

hashes_to_process = [
    h for h in ptx_tx_hashes if h not in processed_hashes]
print(f'Resolving {len(hashes_to_process):,} remaining tx hashes...')

for i, tx_hash in enumerate(tqdm(hashes_to_process)):
    addr = resolve_tx_hash(tx_hash)
    processed_hashes.add(tx_hash)
    if addr:
        ptx_resolved.add(addr)
    time.sleep(SLEEP)

    if (i + 1) % 200 == 0:
        with open(ptx_ckpt_path, 'w') as f:
            json.dump({
                'processed_hashes': list(processed_hashes),
                'resolved_addrs':   list(ptx_resolved)
            }, f)
        print(f'  Checkpoint: {len(processed_hashes):,} processed, '
              f'{len(ptx_resolved):,} resolved')

# Final save
with open(ptx_ckpt_path, 'w') as f:
    json.dump({
        'processed_hashes': list(processed_hashes),
        'resolved_addrs':   list(ptx_resolved)
    }, f)
print(f'Resolution complete: {len(ptx_resolved):,} unique addresses')

# Add resolved addresses to phishing contract pool
# Exclude any that are already confirmed benign
ptx_resolved_clean = ptx_resolved - bc_raw - bw_raw
pc_raw.update(ptx_resolved_clean)

# Re-clean all pools after updating pc_raw
pw_raw -= pc_raw
bw_raw -= pc_raw
bc_raw -= pc_raw

print(f'Phishing contracts after resolution: {len(pc_raw):,}')
print(f'Phishing wallets after re-clean:     {len(pw_raw):,}')

assert len(pw_raw & pc_raw) == 0, 'pw & pc after resolution'
assert len(pc_raw & bc_raw) == 0, 'pc & bc after resolution'
print('Post-resolution contamination check: PASSED')


In [ ]:
# Cell 6: API utility functions
def get_tx_count(address, min_count):
    '''
    Check if address has at least min_count transactions.
    Returns True if result list length >= min_count, False otherwise.
    '''
    try:
        r = requests.get(ETHERSCAN_BASE, params={
            'chainid':    ETHERSCAN_CHAIN_ID,
            'module':     'account',
            'action':     'txlist',
            'address':    address,
            'startblock': 0,
            'endblock':   99999999,
            'offset':     min_count,
            'sort':       'desc',
            'page':       1,
            'apikey':     ETHERSCAN_API_KEY
        }, timeout=10)
        data = r.json()
        if data.get('status') == '1' and data.get('result'):
            return len(data['result']) >= min_count
        return False
    except Exception:
        return False


def get_bytecode(address):
    try:
        r = requests.get(ETHERSCAN_BASE, params={
            'chainid': ETHERSCAN_CHAIN_ID,
            'module':  'proxy',
            'action':  'eth_getCode',
            'address': address,
            'tag':     'latest',
            'apikey':  ETHERSCAN_API_KEY
        }, timeout=10)
        time.sleep(SLEEP)
        result = r.json().get('result', '0x')
        # Validate: real bytecode is hex starting with 0x
        if isinstance(result, str) and result.startswith('0x'):
            return result if result else '0x'
        return '0x'
    except Exception:
        try:
            raw = web3_primary.eth.get_code(Web3.to_checksum_address(address))
            return raw.hex() if raw else '0x'
        except Exception:
            return '0x'

print('API utility functions defined.')


In [ ]:
# Cell 7: Activity filter — builds 4 candidate lists
# Phishing wallets: need 2200 passing (2000 target + 10% buffer)
# Benign wallets:   need 2200 passing
# Phishing contracts: collect ALL that pass (no early stop)
# Benign contracts: collect phishing_count * 1.10 + 50

CANDIDATES_CKPT = f'{DRIVE_BASE}/data/checkpoints/candidates.json'

# If checkpoint exists, load and skip filtering entirely
if os.path.exists(CANDIDATES_CKPT):
    with open(CANDIDATES_CKPT) as f:
        cands = json.load(f)
    pw_candidates = cands['pw_candidates']
    bw_candidates = cands['bw_candidates']
    pc_candidates = cands['pc_candidates']
    bc_candidates = cands['bc_candidates']
    print('Candidate lists loaded from checkpoint — skipping activity filter')
    print(f'  Phishing wallets:   {len(pw_candidates):,}')
    print(f'  Benign wallets:     {len(bw_candidates):,}')
    print(f'  Phishing contracts: {len(pc_candidates):,}')
    print(f'  Benign contracts:   {len(bc_candidates):,}')
else:
    # ── Phishing wallets ─────────────────────────────────────────
    pw_list = list(pw_raw)
    random.shuffle(pw_list)
    pw_candidates, pw_failed = [], []
    TARGET_PW = 2200

    print(f'Filtering {len(pw_list):,} phishing wallets (need {TARGET_PW})...')
    for i, addr in enumerate(pw_list):
        if len(pw_candidates) >= TARGET_PW:
            break
        time.sleep(SLEEP)
        if get_tx_count(addr, WALLET_ACTIVITY_MIN):
            pw_candidates.append(addr)
        else:
            pw_failed.append(addr)
        if (i + 1) % 100 == 0:
            print(f'  [{i+1:,}/{len(pw_list):,}] passed: {len(pw_candidates):,} | failed: {len(pw_failed):,}')
    print(f'  DONE — Passed: {len(pw_candidates):,} | Failed: {len(pw_failed):,}')
    if len(pw_candidates) < 2000:
        print(f'  WARNING: Only {len(pw_candidates)} passed.')

    # ── Benign wallets ───────────────────────────────────────────
    bw_list = list(bw_raw)
    random.shuffle(bw_list)
    bw_candidates, bw_failed = [], []
    TARGET_BW = 2200

    print(f'Filtering {len(bw_list):,} benign wallets (need {TARGET_BW})...')
    for i, addr in enumerate(bw_list):
        if len(bw_candidates) >= TARGET_BW:
            break
        time.sleep(SLEEP)
        if get_tx_count(addr, WALLET_ACTIVITY_MIN):
            bw_candidates.append(addr)
        else:
            bw_failed.append(addr)
        if (i + 1) % 100 == 0:
            print(f'  [{i+1:,}/{len(bw_list):,}] passed: {len(bw_candidates):,} | failed: {len(bw_failed):,}')
    print(f'  DONE — Passed: {len(bw_candidates):,} | Failed: {len(bw_failed):,}')

    # ── Phishing contracts ───────────────────────────────────────
    pc_list = list(pc_raw)
    random.shuffle(pc_list)
    pc_candidates, pc_failed = [], []

    print(f'Filtering {len(pc_list):,} phishing contracts (collecting all)...')
    for i, addr in enumerate(pc_list):
        time.sleep(SLEEP)
        if get_tx_count(addr, CONTRACT_ACTIVITY_MIN):
            pc_candidates.append(addr)
        else:
            pc_failed.append(addr)
        if (i + 1) % 100 == 0:
            print(f'  [{i+1:,}/{len(pc_list):,}] passed: {len(pc_candidates):,} | failed: {len(pc_failed):,}')
    print(f'  DONE — Passed: {len(pc_candidates):,} | Failed: {len(pc_failed):,}')

    # ── Benign contracts ─────────────────────────────────────────
    TARGET_BC = int(len(pc_candidates) * 1.10) + 50
    bc_list   = list(bc_raw)
    random.shuffle(bc_list)
    bc_candidates, bc_failed = [], []

    bc_fail_activity = 0
    bc_fail_bytecode = 0
    bc_fail_verified = 0
    bc_pass          = 0

    print(f'Filtering benign contracts (need {TARGET_BC}) — activity + bytecode + verified...')
    for i, addr in enumerate(bc_list):
        if len(bc_candidates) >= TARGET_BC:
            break

        time.sleep(SLEEP)

        # Check 1: minimum activity
        if not get_tx_count(addr, CONTRACT_ACTIVITY_MIN):
            bc_failed.append(addr)
            bc_fail_activity += 1
        else:
            # Check 2: live bytecode
            time.sleep(SLEEP)
            bytecode = get_bytecode(addr)
            if bytecode == '0x':
                bc_failed.append(addr)
                bc_fail_bytecode += 1
            else:
                # Check 3: verified source code on Etherscan
                try:
                    r = requests.get(ETHERSCAN_BASE, params={
                        'chainid': ETHERSCAN_CHAIN_ID,
                        'module':  'contract',
                        'action':  'getsourcecode',
                        'address': addr,
                        'apikey':  ETHERSCAN_API_KEY
                    }, timeout=15)
                    time.sleep(SLEEP)
                    result      = r.json().get('result', [])
                    is_verified = 1 if (result and result[0].get('SourceCode', '').strip()) else 0
                except Exception:
                    is_verified = 0

                if not is_verified:
                    bc_failed.append(addr)
                    bc_fail_verified += 1
                else:
                    bc_candidates.append(addr)
                    bc_pass += 1

        if (i + 1) % 100 == 0:
            print(f'  [{i+1:,}/{len(bc_list):,}] passed: {bc_pass:,} | '
                  f'fail_activity: {bc_fail_activity:,} | '
                  f'fail_bytecode: {bc_fail_bytecode:,} | '
                  f'fail_verified: {bc_fail_verified:,}')

    print(f'  DONE — Passed: {bc_pass:,} | Failed activity: {bc_fail_activity:,} | '
          f'Failed bytecode: {bc_fail_bytecode:,} | Failed verified: {bc_fail_verified:,}')

    # ── Force-append DeFi seeds (guarantee inclusion regardless of TARGET_BC) ──
    bc_candidates_set = set(bc_candidates)
    defi_force_added  = 0
    print(f'Force-checking {len(defi_seeds)} DeFi seeds not already in bc_candidates...')
    for addr in defi_seeds:
        if addr in bc_candidates_set:
            continue
        time.sleep(SLEEP)
        # Check 1: minimum activity
        if not get_tx_count(addr, CONTRACT_ACTIVITY_MIN):
            continue
        # Check 2: live bytecode
        time.sleep(SLEEP)
        bytecode = get_bytecode(addr)
        if bytecode == '0x':
            continue
        # Check 3: verified source code on Etherscan
        try:
            r = requests.get(ETHERSCAN_BASE, params={
                'chainid': ETHERSCAN_CHAIN_ID,
                'module':  'contract',
                'action':  'getsourcecode',
                'address': addr,
                'apikey':  ETHERSCAN_API_KEY
            }, timeout=15)
            time.sleep(SLEEP)
            result      = r.json().get('result', [])
            is_verified = 1 if (result and result[0].get('SourceCode', '').strip()) else 0
        except Exception:
            is_verified = 0
        if not is_verified:
            continue
        bc_candidates.append(addr)
        bc_candidates_set.add(addr)
        defi_force_added += 1
    print(f'  DeFi seeds force-added to bc_candidates: {defi_force_added}')

    # Save candidate lists so this cell can be skipped on resume
    with open(CANDIDATES_CKPT, 'w') as f:
        json.dump({
            'pw_candidates': pw_candidates,
            'bw_candidates': bw_candidates,
            'pc_candidates': pc_candidates,
            'bc_candidates': bc_candidates
        }, f)
    print('Candidate lists saved to checkpoint.')

print()
print('CANDIDATE SUMMARY:')
print(f'  Phishing wallets:   {len(pw_candidates):,}')
print(f'  Benign wallets:     {len(bw_candidates):,}')
print(f'  Phishing contracts: {len(pc_candidates):,}')
print(f'  Benign contracts:   {len(bc_candidates):,}')

# Verify candidate lists have zero cross-contamination
pw_s = set(pw_candidates); bw_s = set(bw_candidates)
pc_s = set(pc_candidates); bc_s = set(bc_candidates)
assert len(pw_s & bw_s) == 0, 'pw & bw contamination'
assert len(pw_s & pc_s) == 0, 'pw & pc contamination'
assert len(pc_s & bc_s) == 0, 'pc & bc contamination'
assert len(bw_s & pc_s) == 0, 'bw & pc contamination'
print('Candidate list contamination check: PASSED')


In [ ]:
# Cell 8: Wallet data collection function
def collect_wallet(address):
    '''
    Collect raw wallet data via 5 Etherscan calls.
    Returns dict with 6 data keys, or None if no valid txs found.
    Returns None when txlist fails or all txs have isError == 1.
    Never raises.
    '''
    addr = address.lower()

    # Call 1: txlist — full transaction history
    try:
        r = requests.get(ETHERSCAN_BASE, params={
            'chainid':    ETHERSCAN_CHAIN_ID,
            'module':     'account', 'action': 'txlist',
            'address':    addr,      'startblock': 0,
            'endblock':   99999999,  'offset': 500,
            'sort':       'desc',    'page': 1,
            'apikey':     ETHERSCAN_API_KEY
        }, timeout=15)
        time.sleep(SLEEP)
        data = r.json()
        txs  = ([t for t in data['result'] if t.get('isError') == '0']
                if data.get('status') == '1' else [])
    except Exception:
        time.sleep(SLEEP)
        txs = []

    if not txs:
        return None

    # Call 2: tokentx — ERC-20 token transfers
    try:
        r = requests.get(ETHERSCAN_BASE, params={
            'chainid': ETHERSCAN_CHAIN_ID,
            'module':  'account', 'action': 'tokentx',
            'address': addr,      'offset': 200,
            'sort':    'desc',    'page': 1,
            'apikey':  ETHERSCAN_API_KEY
        }, timeout=15)
        time.sleep(SLEEP)
        data     = r.json()
        tokentxs = data['result'] if data.get('status') == '1' else []
    except Exception:
        time.sleep(SLEEP)
        tokentxs = []

    # Call 3: balance — current ETH balance in wei
    try:
        r = requests.get(ETHERSCAN_BASE, params={
            'chainid': ETHERSCAN_CHAIN_ID,
            'module':  'account', 'action': 'balance',
            'address': addr,      'tag': 'latest',
            'apikey':  ETHERSCAN_API_KEY
        }, timeout=15)
        time.sleep(SLEEP)
        result = r.json().get('result', '0')
        try:
            int(result)
            balance_wei = result
        except Exception:
            balance_wei = '0'
    except Exception:
        time.sleep(SLEEP)
        balance_wei = '0'

    # Call 4: getLogs — ERC-20 approval events where this wallet is owner
    # topic1 pads the 20-byte wallet address to 32 bytes for topic matching
    topic1 = '0x000000000000000000000000' + addr[2:]
    try:
        r = requests.get(ETHERSCAN_BASE, params={
            'chainid':   ETHERSCAN_CHAIN_ID,
            'module':    'logs',   'action': 'getLogs',
            'topic0':    APPROVAL_TOPIC,
            'topic1':    topic1,
            'fromBlock': 0,       'toBlock': 'latest',
            'apikey':    ETHERSCAN_API_KEY
        }, timeout=15)
        time.sleep(SLEEP)
        data              = r.json()
        approval_logs_raw = (data['result']
                             if data.get('status') == '1' else [])
    except Exception:
        time.sleep(SLEEP)
        approval_logs_raw = []

    # Call 5: first-ever transaction — for accurate wallet age
    first_tx_timestamp = None
    try:
        r = requests.get(ETHERSCAN_BASE, params={
            'chainid':    ETHERSCAN_CHAIN_ID,
            'module':     'account', 'action': 'txlist',
            'address':    addr,
            'startblock': 0, 'endblock': 99999999,
            'offset':     1, 'sort': 'asc', 'page': 1,
            'apikey':     ETHERSCAN_API_KEY
        }, timeout=15)
        time.sleep(SLEEP)
        data = r.json()
        if data.get('status') == '1' and data.get('result'):
            first_tx_timestamp = int(data['result'][0]['timeStamp'])
    except Exception:
        time.sleep(SLEEP)

    return {
        'address':            addr,
        'txs_json':           json.dumps(txs),
        'tokentxs_json':      json.dumps(tokentxs),
        'balance_wei':        balance_wei,
        'approval_logs_json': json.dumps(approval_logs_raw),
        'first_tx_timestamp': first_tx_timestamp
    }


In [ ]:
# Cell 9: Contract data collection functions
from web3 import Web3

web3_primary  = Web3(Web3.HTTPProvider(CLOUDFLARE_RPC))
web3_fallback = Web3(Web3.HTTPProvider(ALCHEMY_RPC))

def _fetch_contract_data(addr):
    # Call 1: getabi
    try:
        r = requests.get(ETHERSCAN_BASE, params={
            'chainid': ETHERSCAN_CHAIN_ID,
            'module':  'contract', 'action': 'getabi',
            'address': addr,       'apikey': ETHERSCAN_API_KEY
        }, timeout=15)
        time.sleep(SLEEP)
        data = r.json()
        if data.get('status') == '1':
            abi_json = data['result']
            try:
                json.loads(abi_json)
            except Exception:
                abi_json = '[]'
        else:
            abi_json = '[]'
    except Exception:
        time.sleep(SLEEP)
        abi_json = '[]'

    # Call 2: getsourcecode
    try:
        r = requests.get(ETHERSCAN_BASE, params={
            'chainid': ETHERSCAN_CHAIN_ID,
            'module':  'contract', 'action': 'getsourcecode',
            'address': addr,       'apikey': ETHERSCAN_API_KEY
        }, timeout=15)
        time.sleep(SLEEP)
        result      = r.json().get('result', [])
        is_verified = (1 if result and isinstance(result, list)
                       and result[0].get('SourceCode', '') else 0)
    except Exception:
        time.sleep(SLEEP)
        is_verified = 0

    # Call 3: txlist
    try:
        r = requests.get(ETHERSCAN_BASE, params={
            'chainid':    ETHERSCAN_CHAIN_ID,
            'module':     'account', 'action': 'txlist',
            'address':    addr,      'startblock': 0,
            'endblock':   99999999,  'offset': 500,
            'sort':       'desc',    'page': 1,
            'apikey':     ETHERSCAN_API_KEY
        }, timeout=15)
        time.sleep(SLEEP)
        data = r.json()
        txs  = ([t for t in data['result'] if t.get('isError') == '0']
                if data.get('status') == '1' else [])
    except Exception:
        time.sleep(SLEEP)
        txs = []

    # Call 4: eth_getCode
    bytecode_hex = get_bytecode(addr)

    return bytecode_hex, abi_json, is_verified, txs

def collect_phishing_contract(address):
    '''
    Collect phishing contract data.
    Requires live bytecode (bytecode_hex != 0x) and at least 2 transactions.
    Self-destructed contracts are rejected — v2 uses only contracts with
    live bytecode so features are consistent across both classes.
    Returns dict or None.
    '''
    addr                                     = address.lower()
    bytecode_hex, abi_json, is_verified, txs = _fetch_contract_data(addr)
    if bytecode_hex == '0x':
        return None
    if len(txs) < 2:
        return None
    return {
        'address':      addr,
        'bytecode_hex': bytecode_hex,
        'abi_json':     abi_json,
        'is_verified':  is_verified,
        'txs_json':     json.dumps(txs)
    }

def collect_benign_contract(address):
    '''
    Collect benign contract data.
    Requires: live bytecode, verified source code, and at least 2 transactions.
    Returns dict or None.
    '''
    addr                                     = address.lower()
    bytecode_hex, abi_json, is_verified, txs = _fetch_contract_data(addr)

    # Check 1: live bytecode (existing)
    if bytecode_hex == '0x':
        return None

    # Check 2: verified source code (NEW)
    if is_verified == 0:
        return None

    # Check 3: minimum 2 transactions (existing)
    if len(txs) < 2:
        return None

    return {
        'address':      addr,
        'bytecode_hex': bytecode_hex,
        'abi_json':     abi_json,
        'is_verified':  is_verified,
        'txs_json':     json.dumps(txs)
    }


print('Collection functions defined.')


In [ ]:
# Cell 10: Collect phishing wallets
# Checkpoint allows resume if session disconnects mid-collection.
# Resume: re-run this cell — it loads the checkpoint and continues.
CKPT_PW = f'{DRIVE_BASE}/data/checkpoints/wallet_phishing_checkpoint.json'

if os.path.exists(CKPT_PW):
    with open(CKPT_PW) as f:
        phish_w_collected = json.load(f)
    already_pw = set(r['address'] for r in phish_w_collected)
    print(f'Resumed: {len(phish_w_collected):,} already collected')
else:
    phish_w_collected = []
    already_pw        = set()
    print('Starting fresh phishing wallet collection')

remaining_pw = [a for a in pw_candidates if a not in already_pw]
need_pw      = TARGET_PER_CLASS - len(phish_w_collected)
print(f'Need {need_pw} more from {len(remaining_pw):,} remaining...')
failed_pw = []

for i, addr in enumerate(tqdm(remaining_pw)):
    if len(phish_w_collected) >= TARGET_PER_CLASS:
        break
    result = collect_wallet(addr)
    if result is None:
        failed_pw.append(addr)
        continue
    phish_w_collected.append(result)
    if (i + 1) % 100 == 0:
        with open(CKPT_PW, 'w') as f:
            json.dump(phish_w_collected, f)
        print(f'  Checkpoint: {len(phish_w_collected):,} | '
              f'failed: {len(failed_pw):,}')

with open(CKPT_PW, 'w') as f:
    json.dump(phish_w_collected, f)
print(f'Done: {len(phish_w_collected):,} collected | {len(failed_pw):,} failed')


In [ ]:
# Cell 11: Collect benign wallets
CKPT_BW = f'{DRIVE_BASE}/data/checkpoints/wallet_benign_checkpoint.json'

if os.path.exists(CKPT_BW):
    with open(CKPT_BW) as f:
        benign_w_collected = json.load(f)
    already_bw = set(r['address'] for r in benign_w_collected)
    print(f'Resumed: {len(benign_w_collected):,} already collected')
else:
    benign_w_collected = []
    already_bw         = set()
    print('Starting fresh benign wallet collection')

remaining_bw = [a for a in bw_candidates if a not in already_bw]
need_bw      = TARGET_PER_CLASS - len(benign_w_collected)
print(f'Need {need_bw} more from {len(remaining_bw):,} remaining...')
failed_bw = []

for i, addr in enumerate(tqdm(remaining_bw)):
    if len(benign_w_collected) >= TARGET_PER_CLASS:
        break
    result = collect_wallet(addr)
    if result is None:
        failed_bw.append(addr)
        continue
    benign_w_collected.append(result)
    if (i + 1) % 100 == 0:
        with open(CKPT_BW, 'w') as f:
            json.dump(benign_w_collected, f)
        print(f'  Checkpoint: {len(benign_w_collected):,} | '
              f'failed: {len(failed_bw):,}')

with open(CKPT_BW, 'w') as f:
    json.dump(benign_w_collected, f)
print(f'Done: {len(benign_w_collected):,} collected | {len(failed_bw):,} failed')


In [ ]:
# Cell 12: Combine, balance, shuffle, save wallet CSV
# Load from checkpoints if in-memory lists are missing or empty
if not phish_w_collected:
    ckpt = f'{DRIVE_BASE}/data/checkpoints/wallet_phishing_checkpoint.json'
    with open(ckpt) as f:
        phish_w_collected = json.load(f)
    print(f'Loaded phish_w_collected from checkpoint: {len(phish_w_collected):,}')

if not benign_w_collected:
    ckpt = f'{DRIVE_BASE}/data/checkpoints/wallet_benign_checkpoint.json'
    with open(ckpt) as f:
        benign_w_collected = json.load(f)
    print(f'Loaded benign_w_collected from checkpoint: {len(benign_w_collected):,}')

df_pw = pd.DataFrame(phish_w_collected)
df_bw = pd.DataFrame(benign_w_collected)

# Trim BOTH to the smaller count — guarantees a balanced dataset
# even if one class collection fell short of TARGET_PER_CLASS
wallet_final = min(len(df_pw), len(df_bw), TARGET_PER_CLASS)
df_pw        = df_pw.head(wallet_final).copy()
df_bw        = df_bw.head(wallet_final).copy()

df_pw['label'] = 1
df_bw['label'] = 0

df_wallet = pd.concat([df_pw, df_bw], ignore_index=True)
df_wallet = df_wallet.sample(frac=1, random_state=42).reset_index(drop=True)
df_wallet = df_wallet[[
    'address', 'label', 'txs_json',
    'tokentxs_json', 'balance_wei', 'approval_logs_json', 'first_tx_timestamp'
]]

# Fill missing first_tx_timestamp with 0 (fallback handled in feature engineering)
df_wallet['first_tx_timestamp'] = df_wallet['first_tx_timestamp'].fillna(0).astype(int)

wallet_path = f'{DRIVE_BASE}/data/raw_wallet_data.csv'
df_wallet.to_csv(wallet_path, index=False)
print(f'Saved raw_wallet_data.csv: {len(df_wallet):,} rows '
      f'({wallet_final} phishing + {wallet_final} benign)')


In [ ]:
# Cell 13: Collect ALL phishing contracts — no early stop
CONTRACT_TARGET = 650  # safety fallback for session resume

CKPT_PC = f'{DRIVE_BASE}/data/checkpoints/contract_phishing_checkpoint.json'

if os.path.exists(CKPT_PC):
    with open(CKPT_PC) as f:
        phish_c_collected = json.load(f)
    already_pc = set(r['address'] for r in phish_c_collected)
    print(f'Resumed: {len(phish_c_collected):,} already collected')
else:
    phish_c_collected = []
    already_pc        = set()
    print('Starting fresh phishing contract collection')

remaining_pc = [a for a in pc_candidates if a not in already_pc]
print(f'Collecting {len(remaining_pc):,} remaining phishing contracts...')
failed_pc = []

for i, addr in enumerate(tqdm(remaining_pc)):
    if len(phish_c_collected) >= CONTRACT_TARGET:
        break
    result = collect_phishing_contract(addr)
    if result is None:
        failed_pc.append(addr)
        continue
    phish_c_collected.append(result)
    if (i + 1) % 50 == 0:
        with open(CKPT_PC, 'w') as f:
            json.dump(phish_c_collected, f)
        print(f'  Checkpoint: {len(phish_c_collected):,} collected')

with open(CKPT_PC, 'w') as f:
    json.dump(phish_c_collected, f)

print(f'Done: {len(phish_c_collected):,} phishing contracts collected')
print(f'Benign contracts will match: {len(phish_c_collected):,}')


In [ ]:
# Cell 14: Collect benign contracts using collect_benign_contract
CONTRACT_TARGET = 650  # safety fallback for session resume

CKPT_BC = f'{DRIVE_BASE}/data/checkpoints/contract_benign_checkpoint.json'

if os.path.exists(CKPT_BC):
    with open(CKPT_BC) as f:
        benign_c_collected = json.load(f)
    already_bc = set(r['address'] for r in benign_c_collected)
    print(f'Resumed: {len(benign_c_collected):,} already collected')
else:
    benign_c_collected = []
    already_bc         = set()
    print('Starting fresh benign contract collection')

# DeFi seeds go first — guaranteed collection before random pool
defi_priority = [a.lower() for a in defi_seeds
                 if a.lower() not in already_bc]

# Remaining pool after DeFi seeds
already_tried = set(bc_candidates) | set(defi_priority)
bc_raw_extras = [a for a in bc_raw if a not in already_tried]
random.shuffle(bc_raw_extras)
full_pool     = defi_priority + \
                [a for a in bc_candidates if a not in already_bc
                 and a.lower() not in set(defi_priority)] + \
                [a for a in bc_raw_extras  if a not in already_bc]

print(f'DeFi priority contracts at front of pool: {len(defi_priority)}')
print(f'Search pool: {len(full_pool):,} addresses')
print(f'Collecting benign contracts (target: {CONTRACT_TARGET:,})...')
failed_bc  = []
skipped_bc = 0

for i, addr in enumerate(tqdm(full_pool)):
    if len(benign_c_collected) >= CONTRACT_TARGET:
        break

    time.sleep(SLEEP)
    result = collect_benign_contract(addr)

    if result is None:
        skipped_bc += 1
        continue

    benign_c_collected.append(result)

    if (i + 1) % 50 == 0:
        with open(CKPT_BC, 'w') as f:
            json.dump(benign_c_collected, f)
        print(f'  Checkpoint: {len(benign_c_collected):,} collected | '
              f'skipped/failed: {skipped_bc:,}')

with open(CKPT_BC, 'w') as f:
    json.dump(benign_c_collected, f)
print(f'Done: {len(benign_c_collected):,} collected | '
      f'skipped/failed: {skipped_bc:,}')


In [ ]:
# Cell 15: Combine, balance, shuffle, save contract CSV
if not phish_c_collected:
    ckpt = f'{DRIVE_BASE}/data/checkpoints/contract_phishing_checkpoint.json'
    with open(ckpt) as f:
        phish_c_collected = json.load(f)
    print(f'Loaded phish_c_collected from checkpoint: {len(phish_c_collected):,}')

if not benign_c_collected:
    ckpt = f'{DRIVE_BASE}/data/checkpoints/contract_benign_checkpoint.json'
    with open(ckpt) as f:
        benign_c_collected = json.load(f)
    print(f'Loaded benign_c_collected from checkpoint: {len(benign_c_collected):,}')

df_pc = pd.DataFrame(phish_c_collected)
df_bc = pd.DataFrame(benign_c_collected)

contract_final = min(len(df_pc), len(df_bc), CONTRACT_TARGET)
df_pc          = df_pc.head(contract_final).copy()
df_bc          = df_bc.head(contract_final).copy()

df_pc['label'] = 1
df_bc['label'] = 0

df_contract = pd.concat([df_pc, df_bc], ignore_index=True)
df_contract = df_contract.sample(frac=1, random_state=42).reset_index(drop=True)
df_contract = df_contract[[
    'address', 'label', 'bytecode_hex',
    'abi_json', 'is_verified', 'txs_json'
]]

contract_path = f'{DRIVE_BASE}/data/raw_contract_data.csv'
df_contract.to_csv(contract_path, index=False)
print(f'Saved raw_contract_data.csv: {len(df_contract):,} rows '
      f'({contract_final} phishing + {contract_final} benign)')

assert len(df_contract) == CONTRACT_TARGET * 2,              f'Wrong total: {len(df_contract)}'
assert (df_contract['label']==1).sum() == CONTRACT_TARGET,   'Phishing count wrong'
assert (df_contract['label']==0).sum() == CONTRACT_TARGET,   'Benign count wrong'
assert df_contract['address'].nunique() == CONTRACT_TARGET * 2, 'Duplicate addresses found'
assert df_contract.isnull().sum().sum() == 0,                'Nulls found'
print('Contract CSV validation passed.')



In [ ]:
# Cell 16: Final validation — 24 checks across both output files
import json as json_lib

VALID_ADDR   = r'^0x[0-9a-fA-F]{40}$'
results      = []
report_lines = []

def check(name, condition, detail=''):
    status = 'PASS' if condition else 'FAIL'
    line   = f'[{status}] {name}'
    if detail and not condition:
        line += f' — {detail}'
    print(line)
    results.append(condition)
    report_lines.append(line)

# Reload from disk to validate the saved files, not in-memory dataframes
df_w = pd.read_csv(wallet_path)
df_c = pd.read_csv(contract_path)

print('WALLET FILE')
print('-'*40)
nw = len(df_w)
pw = (df_w['label']==1).sum()
bw = (df_w['label']==0).sum()
check('Row count is even',      nw % 2 == 0, f'rows={nw}')
check('Labels balanced',        pw == bw,    f'phishing={pw} benign={bw}')
check('At least 3000 rows',     nw >= 3000,  f'rows={nw}')
check('No null values',         df_w.isnull().sum().sum() == 0)
check('No duplicate addresses', df_w['address'].nunique() == nw)
check('Address format valid',   df_w['address'].str.match(VALID_ADDR).all())
check('Columns correct',        df_w.columns.tolist() == [
    'address','label','txs_json',
    'tokentxs_json','balance_wei','approval_logs_json','first_tx_timestamp'])
try:
    df_w['txs_json'].apply(json_lib.loads)
    check('txs_json parseable', True)
except Exception as e:
    check('txs_json parseable', False, str(e))
try:
    df_w['approval_logs_json'].apply(json_lib.loads)
    check('approval_logs_json parseable', True)
except Exception as e:
    check('approval_logs_json parseable', False, str(e))
txs_lens = df_w['txs_json'].apply(lambda x: len(json_lib.loads(x)))
check('All wallets have >= 1 valid tx', (txs_lens >= 1).all(),
      f'{(txs_lens < 1).sum()} wallets have 0 txs')

print()
print('CONTRACT FILE')
print('-'*40)
nc = len(df_c)
pc = (df_c['label']==1).sum()
bc = (df_c['label']==0).sum()
check('Row count is even',      nc % 2 == 0)
check('Labels balanced',        pc == bc, f'phishing={pc} benign={bc}')
check('At least 400 rows',      nc >= 400, f'rows={nc}')
check('No null values',         df_c.isnull().sum().sum() == 0)
check('No duplicate addresses', df_c['address'].nunique() == nc)
check('Address format valid',   df_c['address'].str.match(VALID_ADDR).all())
check('Columns correct',        df_c.columns.tolist() == [
    'address','label','bytecode_hex',
    'abi_json','is_verified','txs_json'])
check('bytecode_hex starts 0x', df_c['bytecode_hex'].str.startswith('0x').all())
check('is_verified only 0 or 1', df_c['is_verified'].isin([0,1]).all())
try:
    df_c['abi_json'].apply(json_lib.loads)
    df_c['txs_json'].apply(json_lib.loads)
    check('Contract JSON parseable', True)
except Exception as e:
    check('Contract JSON parseable', False, str(e))
benign_c = df_c[df_c['label']==0]
check('Benign contracts all have active bytecode',
      (benign_c['bytecode_hex'] != '0x').all(),
      f"{(benign_c['bytecode_hex']=='0x').sum()} have no bytecode")

print()
print('CROSS-FILE CHECKS')
print('-'*40)
w_addrs = set(df_w['address'])
c_addrs = set(df_c['address'])
check('No address in both files', len(w_addrs & c_addrs) == 0,
      f'{len(w_addrs & c_addrs)} overlapping')
pw_set = set(df_w[df_w['label']==1]['address'])
bw_set = set(df_w[df_w['label']==0]['address'])
pc_set = set(df_c[df_c['label']==1]['address'])
bc_set = set(df_c[df_c['label']==0]['address'])
check('No phishing wallet in benign contracts', len(pw_set & bc_set) == 0)
check('No benign wallet in phishing contracts', len(bw_set & pc_set) == 0)

print()
print('='*50)
all_pass   = all(results)
fail_count = sum(1 for r in results if not r)
if all_pass:
    print('ALL 24 CHECKS PASSED')
    print('READY FOR FEATURE ENGINEERING')
else:
    print(f'{fail_count} CHECKS FAILED — resolve before proceeding')


In [ ]:
# Cell 17: Save report and print final statistics
import datetime
timestamp   = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
report_path = f'{DRIVE_BASE}/data/validation_report.txt'

with open(report_path, 'w') as f:
    f.write('PhishGuard Dataset Validation Report\n')
    f.write(f'Generated: {timestamp}\n')
    f.write('='*50 + '\n\n')
    f.write(f'Wallet CSV:   {len(df_w):,} rows\n')
    f.write(f'Contract CSV: {len(df_c):,} rows\n\n')
    for line in report_lines:
        f.write(line + '\n')
    f.write('\n')
    f.write('RESULT: ALL PASSED\n' if all_pass
            else 'RESULT: FAILURES DETECTED\n')
print(f'Report saved to {report_path}')

print()
print('FINAL DATASET STATISTICS')
print('='*50)
print()
print('WALLET MODEL TRAINING DATA')
print(f'  Total rows:             {len(df_w):,}')
print(f'  Phishing (label=1):     {(df_w["label"]==1).sum():,}')
print(f'  Benign   (label=0):     {(df_w["label"]==0).sum():,}')
txs_p = df_w['txs_json'].apply(lambda x: len(json_lib.loads(x)))
print(f'  Avg txs per wallet:     {txs_p.mean():.1f}')
approvals = df_w['approval_logs_json'].apply(
    lambda x: len(json_lib.loads(x)))
print(f'  Wallets with approvals: {(approvals > 0).sum():,}')
print()
print('CONTRACT MODEL TRAINING DATA')
print(f'  Total rows:             {len(df_c):,}')
print(f'  Phishing (label=1):     {(df_c["label"]==1).sum():,}')
print(f'  Benign   (label=0):     {(df_c["label"]==0).sum():,}')
verified  = (df_c['is_verified']==1).sum()
self_dest = (df_c[df_c['label']==1]['bytecode_hex'] == '0x').sum()
print(f'  Verified contracts:     {verified:,} ({verified/len(df_c)*100:.1f}%)')
print(f'  Self-destructed phishing kept: {self_dest:,}')
print()
print('OUTPUT FILES SAVED TO DRIVE:')
print(f'  {DRIVE_BASE}/data/raw_wallet_data.csv')
print(f'  {DRIVE_BASE}/data/raw_contract_data.csv')
print()
print('NEXT STEP: Notebook 02 — Wallet Feature Engineering')
